# 1. Introduction

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>1.1 Objectives</b></p>
</div>

I'm very excited to participate in kaggle's first **unsupervised clustering** TPS competition. The goal is to **predict** the cluster each sample belongs to. However, we are not even given the number of clusters there should be beforehand. 

I will try to answer the following questions:
* *How **many clusters** should we use?*
* *What is the **competition metric** and where does it come from?*
* *What is the **best model** for the data*?
* *How do we **ensemble** predictions together?*

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>1.2 Libraries</b></p>
</div>

In [ ]:
# Core
import numpy as np
import pandas as pd
import seaborn as sns
sns.set(style='darkgrid', font_scale=1.4)
import matplotlib.pyplot as plt
%matplotlib inline
from itertools import combinations
import math
import statistics
from scipy import stats
from scipy.stats import pearsonr
from scipy.stats import shapiro
from scipy.stats import chi2
from scipy.stats import poisson
import time
from datetime import datetime
import matplotlib.dates as mdates
import plotly.express as px
from termcolor import colored
import warnings
warnings.filterwarnings("ignore")

# Sklearn
import sklearn
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture

# UMAP
import umap
import umap.plot

# 2. Data

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>2.1 Load data</b></p>
</div>

* There are **29** features, all of them masked.
* There are **almost 100,000** data points.

In [ ]:
# Save to df
data=pd.read_csv('../input/tabular-playground-series-jul-2022/data.csv', index_col='id')

# Shape and preview
print('Dataframe shape:', data.shape)
data.head()

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>2.2 Missing values</b></p>
</div>

There are **no missing values**.

In [ ]:
print('MISSING VALUES:')
print(data.isna().sum().sum())

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>2.3 Duplicates</b></p>
</div>

There are **no duplicated values**.

In [ ]:
print(f'Duplicates in dataset: {data.duplicated().sum()}, ({np.round(100*data.duplicated().sum()/len(data),1)}%)')

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>2.4 Data types</b></p>
</div>

There are **7 discrete** features and **22 continuous** features.

In [ ]:
data.dtypes

# 3. EDA

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>3.1 Discrete features</b></p>
</div>

* There are **7** discrete features: *f_07* to *f_13*.
* Values are **non-negative**. 
* Distributions are all similar, perhaps **Poisson**.

In [ ]:
# Figure with subplots
fig=plt.figure(figsize=(15,14))

for i in range(7):
    # New subplot
    plt.subplot(4,2,i+1)
    feat_num=i+7
    sns.countplot(x=data.iloc[:,feat_num])
    
    # Aesthetics
    plt.title(f'Feature: 0{feat_num}')
    plt.xlim([-1,44])      # same scale for all plots
    plt.ylim([0,11000])   # same scale for all plots
    plt.xticks(np.arange(0,44,2))
    plt.xlabel('')
    
# Overall aesthetics
fig.suptitle('Discrete feature distributions',  size=20)
fig.tight_layout()  # Improves appearance a bit
plt.show()

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>3.2 Continuous features</b></p>
</div>

* There are **22** continuous features: *f_00* to *f_06* and *f_14* to *f_28*
* Distributions are all **Normal**, usually with mean 0 and standard deviation 1.
* Values typically lie between -5 and +5.

In [ ]:
# Continuous features
cont_feats=[f'f_0{i}' for i in range(7)]
cont_feats=cont_feats + [f'f_{i}' for i in range(14,29)]

# Figure with subplots
fig=plt.figure(figsize=(15,14))

for i, f in enumerate(cont_feats):
    # New subplot
    plt.subplot(6,4,i+1)
    sns.histplot(x=data[f])
    
    # Aesthetics
    plt.title(f'Feature: {f}')
    plt.xlabel('')
    
# Overall aesthetics
fig.suptitle('Continuous feature distributions',  size=20)
fig.tight_layout()  # Improves appearance a bit
plt.show()

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>3.3 Hypothesis testing</b></p>
</div>

**Shapiro-Wilk Test** (Copied from [Francisco Javier Gallego & Torch me](https://www.kaggle.com/code/javigallego/outliers-eda-clustering-tutorial))

This test is used to test whether a dataset is distributed **normally** or not. The null hypothesis is that a sample $$x_1\hspace{0.1cm},\hspace{0.1cm}\cdots\hspace{0.1cm},\hspace{0.1cm}x_n$$ comes from a normally distributed population. It was published in 1965 by Samuel Shapiro and Martin Wilk and **is considered to be one of the most powerful tests for normality testing.** The test statistic is 

$$W = \frac{(\sum_{i=1}^{n}a_{i}x_i)^2}{\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

where

* $x_i$ is the number from the i-th data point (where the sample is ordered from smallest to largest).
* $\bar{x}$ is the sample mean. 
* Variables $a_i$ are calculated via

$$(a_1, ... , a_n) = \frac{m^T V^{-1}}{(m^T V^{-1}V^{-1}m)^{1/2}} \hspace{2cm}m = (m_1 , ... , m_n)$$

where $m_1 , ... , m_n$ are the mean values of the ordered statistic, of independent and identically distributed random variables, sampled from normal distributions and $V$ denotes the covariance matrix of that order statistic. **The null hypothesis is rejected if W is too small. The value of W can range from 0 to 1.**

In [ ]:
# Univariate normality test
for col in data.columns:
    stat, p_value = shapiro(data[col])
    alpha = 0.05    # significance level
    if p_value > alpha: 
        result = colored('Accepted', 'green')
    else:
        result = colored('Rejected','red')        
    print('Feature: {}\t Hypothesis: {}'.format(col, result))

<hr>

**Poisson Dispersion Test**

This test is used to determine whether a feature is distributed according to a **Poisson** distribution or not. The null hypothesis is that 

$$
X_i \sim Po(\lambda) \quad \text{for every i=1, $\ldots$, n}
$$

This is the **most common** test used for verifying a Poisson distribution. The test statistic (called dispersion) is

$$
D = \sum_{i=1}^{n} \frac{(X_i - \bar{X})^2}{\bar{X}},
$$

where

* $X_i$ is the number from the i-th sample point (order doesn't matter)
* $\bar{X}$ is the sample mean.

Note that the **expected value** of this statistic is $\mathbb{E}(D) = \frac{(n-1) Var(X_i)}{E(X_i)} = \frac{(n-1) \lambda}{\lambda}  = n-1$, since the mean and variance of a Poisson distribution is the rate $\lambda$. If $D$ is too '**far away**' from the expected value of $n-1$, then we **reject** the null hypothesis. 

More formally, $D$ has a **chi-squared** distribution with $n-1$ **degrees of freedom** under the null hypothesis. We determine the **critical values** by using a **two-tailed** test with significance level $\alpha=5\%$.

In [ ]:
# Discrete features to test
int_feats = ['f_07', 'f_08', 'f_09', 'f_10', 'f_11', 'f_12', 'f_13']

# Univariate poisson test
for col in data[int_feats].columns:
    # Parameters
    alpha = 0.05                  # significance level
    n = len(data[col])            # sample size
    df = n-1                      # degrees of freedom
    
    # Statistics
    mu = data[col].mean()               # sample mean
    D = ((data[col]-mu)**2).sum()/mu    # test statistic
    
    # Two-tailed test
    q_lower = alpha/2
    q_upper = (1-alpha)/2
    
    # percentile point function = inverse of cdf
    chi2_crit_lower = chi2.ppf(q_lower, df)
    chi2_crit_upper = chi2.ppf(q_upper, df)
    
    if (D<chi2_crit_lower) or (D>chi2_crit_upper):
        result = colored('Rejected', 'red')
    else:
        result = colored('Accepted', 'green')
    print('Feature: {}\t Hypothesis: {}'.format(col, result))
    #print('D:',int(D),', chi2_crit_lower:',int(chi2_crit_lower),', chi2_crit_upper:',int(chi2_crit_upper),'\n')

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>3.4 Q-Q plots</b></p>
</div>

Q-Q plots, aka **Quantile-Quantile** plots, are used to **visually compare** how similar two distributions are to each other. They consist of plotting the quantiles (i.e. regular intervals) of the **observed** distribution against the quantiles of the **theoretical** distribution. The closer the Q-Q plots are to forming a **straight line**, the more confident you can be that the observed and theoretical distributions are the **same**. 

**Normal Q-Q plots**

In [ ]:
# Normal Q-Q plots
figure = plt.figure(figsize = (16,12))
for i in range(len(data.columns)):
    
    # Q-Q plot
    ax = plt.subplot(6,5, i+1)
    stats.probplot(data.iloc[:,i], dist='norm', plot=plt)
    
    # Aesthetics
    ax.get_lines()[0].set_markersize(6.0)
    ax.get_lines()[1].set_linewidth(3.0)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    plt.title(data.columns[i])
    
figure.tight_layout(h_pad=1.0, w_pad=0.5)
plt.suptitle('Normal Q-Q Charts', y=1.02, fontsize=20)
plt.show()

Even though the features *f_22* to *f_28* failed the Shapiro-Wilk test, they still appear to be quite close to being normally distributed. This behaviour could be because these features are made up of a **mixture** of normal distributions. There is not an easy way to verify this however.

<br>
<hr>

**Poisson Q-Q plots**

In [ ]:
# Poisson Q-Q plots
figure = plt.figure(figsize = (16,5))
for i, col in enumerate(int_feats):
    
    # Q-Q plot
    ax = plt.subplot(2, 4, i+1)
    mu = data[col].mean()
    stats.probplot(data[col], dist='poisson', sparams=(mu,), plot=plt)
    
    # Aesthetics
    ax.get_lines()[0].set_markersize(6.0)
    ax.get_lines()[1].set_linewidth(3.0)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    plt.title(col)
    
figure.tight_layout(h_pad=1.0, w_pad=0.5)
plt.suptitle('Poisson Q-Q Charts', y=1.02, fontsize=20)
plt.show()

We can see more clearly that these features are not distributed according to independent Poisson distributions. However, some of them are quite close. It could be also that these features are made up of a **mixture** of Poisson distributions. Unfortunately, there isn't an easy way to verify this.

<br>
<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>3.5 Correlations</b></p>
</div>

* Features *f_00* to *f_06* and *f_14* to *f_21* are **independent** of all other features.
* Discrete features (*f_07* to *f_13*) and features *f_22* to *f_28* are **weakly dependent** of each other.

In [ ]:
# Heatmap of correlations
plt.figure(figsize=(7,5))
sns.heatmap(data.corr().abs(), cmap='Greens', vmin=0, vmax=1)
plt.title('Absolute correlations')

# 4. Elbow method

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>4.1 How it works</b></p>
</div>


The **elbow method** is a practical way to determine the number of clusters in a dataset. It works by plotting the **inertia** (or sometimes distortion) against the **number of clusters**, where **inertia** is defined to be the sum of squared distances of samples to their closest cluster center, i.e. a measure of the models bias.  The '**elbow**' (point of sudden flattening) of the curve is then chosen to be the optimal number of clusters in the dataset.

<center>
<img src="https://www.oreilly.com/library/view/statistics-for-machine/9781788295758/assets/995b8b58-06f1-4884-a2a1-f3648428e947.png" width="500">
</center>

The idea is that we want **low inertia** (because that means we have a good model), but not too low otherwise this will lead to **overfitting** (since if k=number of samples then every point is a cluster and the inertia is 0). The elbow usually represents the point of **diminishing returns** and therefore is a good **heuristic** for the optimal number of clusters.

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>4.2 Applying it </b></p>
</div>

See my [discussion post](https://www.kaggle.com/competitions/tabular-playground-series-jul-2022/discussion/335079) where I used 50 clusters and the whole dataset. To save time here, we will just use 30 clusters and 10% of the data.

In [ ]:
%%time

inertias = []
for k in range(1,30):
    km = KMeans(n_clusters=k)
    km.fit(data.iloc[:10000,:])
    inertias.append(km.inertia_)

# Plot inertias
plt.figure(figsize=(16,6))
plt.plot(range(1,30), inertias, 'bx-')
plt.xlabel('Number of clusters, k')
plt.ylabel('Inertia')
plt.title('Elbow method')
plt.show()

It is **hard** to tell exactly what the optimal value for the number of clusters should be since the curve is quite smooth. We will go with **k=7** for now, but it might be worth experimenting with different values of k as well. 

# 5. Competition metric

It is worth spending some time trying to understand the competition metric. This is called the **Adjusted Rand Index (ARI)**. But to do this, we first need to look at the **Rand Index (RI)**.

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>5.1 Rand Index</b></p>
</div>

The Rand Index (named after **William Rand** from 1971) is a measure of **similarity** between the predicted clusters and the ground truth clusters. It looks at whether **pairs** of data points are in the same or different clusters. Let's work through an **example** to see how it works.

$$\Large RI = \frac{a+b}{{n \choose 2}}$$

<center>
<img src="https://i.postimg.cc/3Ng0nRwZ/RI-n5.jpg" width="700">
</center>

First note that there are $n=5$ data points (denoted by greek letters, alpha to epsilon). The prediction is made up of **3 clusters**, whereas the ground truth is made up of **2 clusters**.

The combinatorial **formula** for the total **number of pairs** of data points is given by ${n \choose 2} = \frac{n(n-1)}{2}$. So for $n=5$, there are 10 total pairs. These are:

$\{\alpha, \beta\}, \{\alpha, \gamma\}, \{\alpha, \delta\}, \{\alpha, \epsilon\}, \{\beta, \gamma\}, \{\beta, \delta\}, \{\beta, \epsilon\}, \{\gamma, \delta\}, \{\gamma, \epsilon\}, \{\delta, \epsilon\}$.

<hr>

To work out the Rand Index, we need to calculate **two quantities**:
* $a$ = # pairs in the **same** cluster in the prediction and the **same** cluster in the ground truth.
* $b$ = # pairs in **different** clusters in the prediction and **different** clusters in the ground truth.

This can be a **bit confusing** but for example, the points ${\color{orange} \alpha}, {\color{orange} \beta}$ are in the same cluster in the prediction (orange) and in the same cluster in the ground truth (orange), so the pair $\{{\color{orange} \alpha}, {\color{orange} \beta}\}$ counts towards $a$. On the other hand, the points ${\color{orange} \alpha}, {\color{green} \delta}$ are in different clusters in both the prediction and ground truth (orange, green) so the pair $\{{\color{orange} \alpha}, {\color{green} \delta}\}$ counts towards $b$. 

<hr>

If we continue like this (**check this yourself**), you will find that the pairs $\{{\color{orange} \alpha}, {\color{orange} \beta}\}, \{{\color{green} \delta}, {\color{green} \epsilon}\}$ are in the **same** cluster for both prediction and ground truth so $a=2$ and the pairs $\{{\color{orange} \alpha}, {\color{green} \delta}\}, \{{\color{orange} \alpha}, {\color{green} \epsilon}\}, \{{\color{orange} \beta}, {\color{green} \delta}\}, \{{\color{orange} \beta}, {\color{green} \epsilon}\}, \{{\color{red} \gamma}, {\color{green} \delta}\}, \{{\color{red} \gamma}, {\color{green} \epsilon}\}$ are in **different** clusters for both prediction and ground truth so $b=6$.

Great, so putting the numbers in we find that $RI=\frac{2+6}{10}=0.8$.

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>5.2 Properties of RI</b></p>
</div>

* RI lies **between 0 and 1**. The closer to 1 the better.
* If the prediction is **perfect**, i.e. equal to the ground truth, then **RI = 1**.
* We say a pair of points is in '**agreement**' if they count towards a or b (above), and in '**disagreement**' otherwise. If we pick two points at **random**, RI gives the **probability** that this pair of points is in agreement. (i.e. the predicted 'state' of the pair is 'correct')
* RI is equivalent to **accuracy** when viewed from a **binary classification** problem over the **pairs** of data points. In particular, each pair is either in agreement (1) or in disagreement (0), in which case $a=\text{True Positives} \, (TP)$ and $b=\text{True Negatives} \, (TN)$ so the Rand Index becomes:

$$RI = \frac{TP + TN}{TP + FP + FN + TN} = \, \text{accuracy of pairs}$$
* The only main **downside** to RI is that the **expected value** of RI, $\mathbb{E}(RI)$, isn't the same for different clustering problems. That means, some problems are **easier** to get a good RI score than others so we can't really **compare** RI between different problems. This is where the adjusted RI comes in. 

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>5.3 Adjusted Rand Index</b></p>
</div>

The **Adjusted Rand Index** (ARI) (introduced by **Hubert** and **Arabie** in 1985) is a 'corrected-for-chance' version of the Rand Index. It substracts RI by the expected value of RI for the specific clusterting problem. It then scales this number so that it has a maximum value of 1. 

$$\Large ARI = \frac{RI - \mathbb{E}(RI)}{max(RI)-\mathbb{E}(RI)}$$

**Properties of ARI:**
* It has a **maximum value of 1** but **no minimum** value (it can be negative). 
* If the prediction is **perfect**, i.e. equal to the ground truth, then **ARI = 1**.
* A score of **0**, means the prediction is as good as picking all the clusters at **random**. 
* It is **comparable** between different clustering problems as its expected value is **constant** (0).

<hr>

We know how to work out the RI and also that $max(RI)=1$, but working out the expectation $\mathbb{E}(RI)$ is much **trickier**. Hubert derived the following (rather complicated) **formula** for the entire ARI. See the **appendix** if you are interested to see the derivation.

$$
\large ARI = \frac{ \left. \sum_{ij} \binom{n_{ij}}{2} - \left[\sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2}\right] \right/ \binom{n}{2} }{ \left. \frac{1}{2} \left[\sum_i \binom{a_i}{2} + \sum_j \binom{b_j}{2}\right] - \left[\sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2}\right] \right/ \binom{n}{2} }
$$

**Note:** This is equivalent to the ARI formula above. 

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>5.4 ARI example</b></p>
</div>

First, we start by **formalising** the clustering problem. We denote our dataset with $n$ objects by $S = \{o_1, o_2, \ldots, o_n \}$ (Each element is just a data point). Let's represent the ground truth and predicted clustering by two **partitions**: $X=\{X_1, \ldots, X_r\}$ and $Y=\{Y_1, \ldots, Y_s\}$, respectively.

Continuing from our previous example, $X=\{X_1, X_2\}$ with $X_1=\{{\color{orange} \alpha},{\color{orange} \beta},{\color{orange} \gamma}\}$, $X_2=\{{\color{green} \delta},{\color{green} \epsilon}\}$ and $Y=\{Y_1, Y_2, Y_3\}$ with $Y_1=\{{\color{orange} \alpha},{\color{orange} \beta}\}$, $Y_2=\{{\color{red} \gamma}\}$, $Y_3=\{{\color{green} \delta},{\color{green} \epsilon}\}$.

<hr>

Then we draw a **contingency** table. Each entry, $n_{i,j}$, denotes how many data points there are in common between the ground truth cluster $X_i$ and the predicted cluster $Y_j$. Mathematically, the formula is $n_{i,j} = |X_i \cap Y_j |$, i.e. the **intersection**.

$$ 
\begin{array}{c|cccc|c}
{{} \atop X}\!\diagdown\!^Y &
Y_1&
Y_2&
\cdots&
Y_s&
\text{sums}
\\
\hline
X_1&
n_{11}&
n_{12}&
\cdots&
n_{1s}&
a_1
\\
X_2&
n_{21}&
n_{22}&
\cdots&
n_{2s}&
a_2
\\
\vdots&
\vdots&
\vdots&
\ddots&
\vdots&
\vdots
\\
X_r&
n_{r1}&
n_{r2}&
\cdots&
n_{rs}&
a_r
\\
\hline
\text{sums}&
b_1&
b_2&
\cdots&
b_s&
\end{array}
$$

For example, to work out $n_{11}$, we look at clusters $X_1=\{{\color{orange} \alpha},{\color{orange} \beta},{\color{orange} \gamma}\}$ and $Y_1=\{{\color{orange} \alpha},{\color{orange} \beta}\}$ and find the points which appear in both of them. In this case, there are 2:  $X_1 \cap Y_1 = \{{\color{orange} \alpha},{\color{orange} \beta}\}$ so $n_{11}=2$. If we continue like this (**check this yourself**) we get:

$$ 
\begin{array}{c|ccc|c}
{{} \atop X}\!\diagdown\!^Y &
Y_1&
Y_2&
Y_3&
\text{sums}
\\
\hline
X_1&
2&
1&
0&
3
\\
X_2&
0&
0&
2&
2
\\
\hline
\text{sums}&
2&
1&
2&
\end{array}
$$

<hr>

We are almost there. To work out ARI, we need to calculate these 3 quantities: $\sum_{i,j} {n_{ij} \choose 2}$, $\sum_{i} {a_{i} \choose 2}$, $\sum_{j} {b_{j} \choose 2}$.

First recall the formula ${n \choose 2} = \frac{n(n-1)}{2}$, which we will be using a lot. E.g. ${5 \choose 2} = 10$

1. $\sum_{i,j} {n_{ij} \choose 2} = {2 \choose 2} + {1 \choose 2} + {0 \choose 2} + {0 \choose 2} + {0 \choose 2} + {2 \choose 2} = 1 + 0 + 0 + 0 + 0 + 1 = 2$.

2. $\sum_{i} {a_{i} \choose 2} = {3 \choose 2} + {2 \choose 2} = 3 + 1 = 4$.

3.  $\sum_{j} {b_{j} \choose 2} = {2 \choose 2} + {1 \choose 2} + {2 \choose 2} = 1 + 0 + 1 = 2$.

So if we plug everything into the formula, we get $ARI = \frac{2 - (4 \times 2) / 10}{(4 + 2)/2 - (4 \times 2)/10} = 0.55$. 

The ARI score (0.55) is quite a bit smaller than the RI score (0.80) we got earlier. This is somewhat expected though with such a small clustering problem; since the number of points $n$ is small, the problem is relatively easy so the ARI makes a **large adjustment**.

# 6. Modelling

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>6.1 Scaling</b></p>
</div>

It is always important to scale the data for clustering problems so that it is easier to compare the distance between data points. 

<center>
<img src="https://149695847.v2.pressablecdn.com/wp-content/uploads/2021/09/image-47.png" width="400">
</center>

There are several ways to do this, e.g.

* *StandardScaler*: scales each column independently to have mean 0 and standard deviation 1, by subtracting by the column **mean** and dividing by the column **standard deviation**.
* *RobustScaler*: does the same as above but uses statistics that are **robust to outliers**, i.e. it subtracts by the **median** and divides by the **interquartile range**. 
* *PowerTransformer*: makes columns more gaussian like by **stabilising variance** and **minising skew**. 

In [ ]:
#scaled_data = pd.DataFrame(StandardScaler().fit_transform(data))
#scaled_data = pd.DataFrame(RobustScaler().fit_transform(data))
scaled_data = pd.DataFrame(PowerTransformer().fit_transform(data))

scaled_data.columns = data.columns

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>6.2 k-Means</b></p>
</div>

k-Means is an **iterative** clustering algorithm that works as follows:
1. Choose coordinates (e.g. randomly) for the locations of the k centroids.
2. Group datapoints together by finding the nearast centroid. (There will always be k goups).
3. Calculate the new centre of each centroid by taking the mean position of datapoints in each group.
4. Iterative until the centroids stop moving by a significant amount.

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif" width="300">
</center>

<br>

k-Means is popular because it is a **reliable** and **fast** algorithm. The main downside is that it assumes the clusters are **spherical**, which is not always the case. 

In [ ]:
%%time

# Baseline
model_km = KMeans(n_clusters=7, random_state=0)
preds_km = model_km.fit_predict(scaled_data)

k-Means doesn't end up doing so well on the leaderboard (it scores around ARI=0.23), so we will need more sophisticated models that allow for non-spherical clusters.

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>6.3 Gaussian Mixture Model (GMM)</b></p>
</div>

A Gaussian Mixture Model (GMM) is a clustering algorithm that assumes the data is made up **combination/mixture** of several (multivariate) **Gaussian/Normal distributions**. In contrast to k-Means, data points are assigned a **probability** of belonging to each cluster as opposed to being assigned a single cluster. These probabilities are worked out using the **Expectation-Maximization** (EM) algorithm, which estimates the **parameters** (mean & covariance matrix) of Gaussian distributions using an **iterative Maximum Likelihood Estimation** (MLE) method. At the end of the learning process, data points are assigned to a cluster by choosing the cluster with the **highest probability** out of all of them. 

Note that the sklearn implementation can put restrictions on the type of covariance matrices learned. In particular, they can be **tied**, where all clusters have the same covariance matrix, **diagonal**, where all covariance matrices are diagonal, **spherical**, where the resulting clusters are spherical and **full**, where there are no restrictions on the covariance matrices.

<center>
<img src="https://c.tenor.com/i1rNMdaKd7MAAAAC/gaussian-mixture-models-em-method-math.gif" width="400">
</center>

GMM is a **sophisticated model** that often produces excellent results. The main downsides are that it can be **slow** for large datasets and that it assumes the clusters are **normally distributed**, which isn't always true. 

In [ ]:
%%time

# Baseline
model_gmm = GaussianMixture(n_components=7, random_state=0)
preds_gmm = model_gmm.fit_predict(scaled_data)

GMM performs much better than k-Means (around ARI=0.49 on public leaderboard). 

We can plot the position of the **center** of each cluster to visualise how well each feature is able to **separate** the different clusters.

In [ ]:
# Code from AmbrosM: https://www.kaggle.com/competitions/tabular-playground-series-jul-2022/discussion/334808
plt.figure(figsize=(20,4))
for i in range(model_gmm.means_.shape[0]):
    plt.scatter(np.arange(scaled_data.shape[1]), model_gmm.means_[i])
plt.xticks(ticks=np.arange(scaled_data.shape[1]), labels=scaled_data.columns)
plt.title('Cluster means')
plt.show()

From this plot we can see that the features *f_00* to *f_06* and *f_14* to *f_21* **don't separate** the clusters at all! This means we might as well **drop** these features as they are not helping us in any way.

In [ ]:
%%time

# Drop useless features
drop_feats = [f'f_0{i}' for i in range(7)]
drop_feats = drop_feats + [f'f_{i}' for i in range(14,22)]
scaled_data_crop = scaled_data.drop(drop_feats, axis=1)

# Remake predictions
model_gmm_crop = GaussianMixture(n_components = 7, random_state=0)
preds_gmm_crop = model_gmm_crop.fit_predict(scaled_data_crop)

Dropping the features doesn't change the ARI score too much but it does **speed up** training time by a factor of about 2. 

<br>

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>6.4 Bayesian Gaussian Mixture Model (Bayesian GMM)</b></p>
</div>

A Bayesian Gaussian Mixture Model (BGMM) is very similar to a GMM. It uses the **same assumptions** on the data and the same approach to finding the clusters. The only difference is in the learning algorithm. Instead of Maximum Likelihood Estimation, BGMMs use **Variational Bayesian Estimation**. 

In contrast to traditional frameworks, the Bayesian approach views parameters as **random variables** rather than fixed unknown quantities. It estimates the distribution of these parameters by sampling from the posterior distribution using methods like **Markov Chain Monte Carlo** (MCMC).



In [ ]:
%%time

# Baseline
model_bgmm = BayesianGaussianMixture(n_components=7, covariance_type='full', max_iter=100, n_init=5, init_params='random', random_state=0)
preds_bgmm = model_bgmm.fit_predict(scaled_data_crop)

Bayesian GMM performs the **best** out of all of the models we've tried so far (it scores around ARI=0.59 on the public leaderboard). It does take **longer** to run though.

# 7. Visualise predictions

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>7.1 Label distribution</b></p>
</div>

In [ ]:
# Countplot
plt.figure(figsize=(10,4))
sns.countplot(x=preds_bgmm)
plt.title('Predicted clusters')

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>7.2 Cluster distributions</b></p>
</div>

To get insight into the underlying distributions, we can plot cluster-wise histograms of each of the remaining features.

**Continuous: (f_22 to f_28)**

In [ ]:
# From https://www.kaggle.com/code/ambrosm/tpsjul22-gaussian-mixture-cluster-analysis
fig, axs = plt.subplots(2, 4, figsize=(20, 7))
axs = axs.ravel()
float_columns = ['f_22','f_23','f_24','f_25','f_26','f_27','f_28']
y=preds_bgmm
for ax, f in zip(axs, float_columns):
    for i in range(7):
        h, edges = np.histogram(data[f][y == i], bins=np.linspace(-5, 5, 26))
        ax.plot((edges[:-1] + edges[1:]) / 2, h, label=f"Cluster {i}", lw=3)
    ax.set_title(f)
#axs[-2].axis('off')
axs[-1].axis('off')
plt.suptitle('Histograms of continuous features by cluster', y=1.02, fontsize=28)
fig.tight_layout(h_pad=1.0, w_pad=0.5)
plt.show()

**Discrete: (f_07 to f_13)**

In [ ]:
# From https://www.kaggle.com/code/ambrosm/tpsjul22-gaussian-mixture-cluster-analysis
prop_cycle = plt.rcParams['axes.prop_cycle']

fig, axs = plt.subplots(2, 4, figsize=(20, 7))
axs = axs.ravel()
int_columns = [col for col in data.columns if data[col].dtype == 'int']
for ax, f in zip(axs, int_columns):
    for i in range(7):
        uv, uc = np.unique(data[f][y == i], return_counts=True)
        ax.plot(uv, uc, alpha=1, color=prop_cycle.by_key()['color'][i % 10], lw=3)
    ax.set_title(f)
    #ax.legend()
axs[-1].axis('off')
plt.suptitle('Histograms of discrete features by cluster', y=1.02, fontsize=28)
fig.tight_layout(h_pad=1.0, w_pad=0.5)
plt.show()

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>7.3 Principle Component Analysis (PCA)</b></p>
</div>

**Principal Component Analysis (PCA)** was the first dimensionality reduction technique discovered (by Karl Pearson - yes, the guy from Pearson's correlation coefficient) and dates back to as early as **1901**. It is very popular because it is **fast**, **easy to implement** and **easy to interpret**. 

PCA works by finding a low dimensional subspace that **maximises the variance** of the data in that subspace and performing a **linear projection**. This basically means the data will be as **spread out** as possible, without changing the relationship between the data points. This allows us to find patterns in dimensions we can visualised.

In [ ]:
%%time

# PCA
pca = PCA(n_components=3)
components = pca.fit_transform(scaled_data_crop)

# 3D scatterplot
fig = px.scatter_3d(
    components, x=0, y=1, z=2, color=preds_bgmm, size=0.1*np.ones(len(scaled_data_crop)), opacity = 1,
    title='PCA plot in 3D',
    labels={'0': 'PC 1', '1': 'PC 2', '2': 'PC 3'},
    width=650, height=500
)
fig.show()

**Explained variance** shows how much of the variance/spread of the data is captured in each dimension, i.e. how **important** each additional **principal component** is to the original data representation.

In [ ]:
# PCA
pca_var = PCA()
pca_var.fit(scaled_data_crop)

# Plot
plt.figure(figsize=(10,5))
xi = np.arange(1,1+scaled_data_crop.shape[1], step=1)
yi = np.cumsum(pca_var.explained_variance_ratio_)
plt.plot(xi, yi, marker='o', linestyle='--', color='b')

# Aesthetics
plt.ylim(0.0,1.1)
plt.xlabel('Number of Components')
plt.xticks(np.arange(1,1+scaled_data_crop.shape[1], step=1))
plt.ylabel('Cumulative variance (%)')
plt.title('Explained variance by each component')
plt.axhline(y=1, color='r', linestyle='-')
plt.gca().xaxis.grid(False)

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>7.4 t-SNE</b></p>
</div>

**t-SNE** (pronounced tiz-knee) stands for **t-distributed Stochastic Neighbor Embedding** and was proposed much more recently by Laurens van der Maaten and Geoffrey Hinton in their [2008 paper](https://www.jmlr.org/papers/volume9/vandermaaten08a/vandermaaten08a.pdf). 
This works in a similar way to PCA but has some key differences:
* Firstly, this is a **stochastic method**. So if you run multiple t-SNE plots on the same dataset it can look different.
* Another difference is that this is an **iterative method**. It works by repeatedly moving datapoints closer or further away from each other depending on how 'similar' they are. 
* The new representation is **non-linear**. This makes it harder to interpret but it can be very effective at 'unravelling' highly non-linear data.

The main downside to t-SNE is that is **very slow** compared to the other dimensionality techniques. This is because it makes calculations on a pair-wise basis, which does not scale well with large datasets.

In [ ]:
%%time

# PCA
tsne = TSNE(n_components=3)
components = tsne.fit_transform(scaled_data_crop.iloc[:5000,:])

# 3D scatterplot
fig = px.scatter_3d(
    components, x=0, y=1, z=2, color=preds_bgmm[:5000], size=0.1*np.ones(len(scaled_data_crop.iloc[:5000,:])), opacity = 1,
    title='t-SNE plot in 3D',
    labels={'0': 'comp. 1', '1': 'comp. 2', '2': 'comp. 3'},
    width=650, height=500
)
fig.show()

Even with just 5% of the data it still takes several minutes to run.

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>7.5 UMAP</b></p>
</div>

**UMAP**, which stands for **Uniform Manifold Approximation and Projection** was proposed by Leland McInnes, John Healy and James Melville in their [2018 paper](http://gobie.csb.pitt.edu/SML/umap.pdf).

It is similar to t-SNE in that it learns a non-linear mapping that preserves clusters but its main advantage is that it is **significantly faster**. It also tends to do better at preserving **global structure** of the data compared to t-SNE. 

Reference: https://pair-code.github.io/understanding-umap/ 

In [ ]:
%%time

# UMAP
um = umap.UMAP(n_components=3)
components_umap = um.fit_transform(scaled_data_crop)

# 3D scatterplot
fig = px.scatter_3d(
    components_umap, x=0, y=1, z=2, color=preds_bgmm, size=0.1*np.ones(len(scaled_data_crop)), opacity = 1,
    title='UMAP plot in 3D',
    labels={'0': 'comp. 1', '1': 'comp. 2', '2': 'comp. 3'},
    width=650, height=500
)
fig.show()

UMAP's **connectivity plot** is a weighted graph that gives insight into the representation of the embedding. It basically shows which connections were most important when creating the projection. 

In [ ]:
%%time

# Connectivity plot
um = umap.UMAP()
X_fit = um.fit(scaled_data_crop)
umap.plot.connectivity(X_fit, show_points=True)

# 8. Submission

In [ ]:
sub = pd.read_csv('../input/tabular-playground-series-jul-2022/sample_submission.csv')
sub['Predicted'] = preds_bgmm
sub.to_csv('submission.csv', index=False)

# 9. References

* [Notebok: Understading the competition metric: Adjusted rand Index](https://www.kaggle.com/competitions/tabular-playground-series-jul-2022/discussion/334534) by [Towhidul Tonmoy](https://www.kaggle.com/towhidultonmoy).
* [Paper: Comparing Partitions](https://link.springer.com/content/pdf/10.1007/BF01908075.pdf) by Hubert and Arabie, 1985.
* [Paper: MEASURING AGREEMENT WHEN TWO OBSERVERS CLASSIFY PEOPLE INTO CATEGORIES NOT DEFINED IN ADVANCE](https://bpspsychub.onlinelibrary.wiley.com/doi/abs/10.1111/j.2044-8317.1974.tb00535.x?casa_token=n4nz2gru0rYAAAAA:gl9377Ehcq4eoUyIfpxJ5Z6CotPTTk8QTptWEC1rfkM3Wp9MEk16UDfr-4CRWNIECO7otF-Fp00ux1I) by Brennan and Light, 1974.
* [Notebook: Outliers+EDA+Clustering Tutorial](https://www.kaggle.com/code/javigallego/outliers-eda-clustering-tutorial) by [Francisco Javier Gallego
](https://www.kaggle.com/javigallego).
* [Article: Details of the Adjusted Rand index and Clustering algorithms](https://faculty.washington.edu/kayee/pca/supp.pdf) by Yeung and Ruzzo, 2001.  
* [Notebook: TPS Jul 22 ADVANCED + 2% SOL](https://www.kaggle.com/code/kartushovdanil/tps-jul-22-advanced-2-sol) by [Torch me](https://www.kaggle.com/kartushovdanil).
* [Discussion: Visualizing the seven clusters](https://www.kaggle.com/competitions/tabular-playground-series-jul-2022/discussion/334808) by [AmbrosM](https://www.kaggle.com/ambrosm).
* [Article: POISSON DISPERSION TEST](https://www.itl.nist.gov/div898/software/dataplot/refman1/auxillar/poisdisp.htm) by NIST.
* [Paper: A Test for the Poisson Distribution](http://www-stat.wharton.upenn.edu/~lbrown/Papers/2002c%20A%20new%20test%20for%20the%20Poisson%20distribution%20(with%20L.%20H.%20Zhao).pdf) by Lawrence D. Brown and Linda H. Zhao, 2002.
* [Lecture notes: Stats 200  Introduction to Statistical Inference](https://artowen.su.domains/courses/200/lec11.pdf) by  Art B. Owen, 2018.

# 10. Appendix

<div style="color:white;display:fill;
            background-color:#506f3f;font-size:150%;
            font-family:Nexa;letter-spacing:0.5px">
    <p style="padding: 4px;color:white;"><b>ARI derivation</b></p>
</div>

If you're like me, then you will be **curious** as to where **Hubert's formula** for ARI even comes from. I spent several hours reading the **original** papers (Hubert & Arabie 1985, Brennan & Light 1974) to try to understand the derivation and I wanted to **share** my findings. Feel free to **skip** this section though, as it is going to be **very mathematical**. 

<hr>

We start by extending the definitions we had before.

* $a$ = # pairs in the **same** cluster in the prediction and the **same** cluster in the ground truth.
* $b$ = # pairs in **different** clusters in the prediction and **different** clusters in the ground truth.
* $c$ = # pairs in **different** clusters in the prediction and in the **same** cluster in the ground truth.
* $d$ = # pairs in the **same** cluster in the prediction and **different** clusters in the ground truth.

Note: since $a+b+c+d= {n \choose 2}$ (total number of pairs), the RI can also be expressed as $RI=\frac{a+b}{a+b+c+d}$.

<hr>

To derive **Hubert's formula**, we need to calcuate $a, b, c, d$ in terms of the **contingency table**. This will allow us to calculate the expected RI later. Recall $n_{ij}=|X_i \cap Y_j|$, so ${n_{ij} \choose 2}$ gives the number of pairs of points that are both in $X_i$ and $Y_j$. By definition, these pairs will be in the **same** cluster ($X_i$) in the prediction and the **same** cluster ($Y_j$) in the ground truth they count towards $a$. To actually calculate $a$, all we need to do is **sum** these combinations over all the **intersections**. That is,

$$a = \sum_{i,j} {n_{ij} \choose 2}$$

<hr>

We can calculate $b$ using a **trick** once we have $a,c,d$. For now, we focus on probably the most **difficult** part, i.e. calculating $c$. We need to work out how many pairs there are that are in the **same** cluster in the ground truth and in **different** clusters in the prediction. This will involve a technique called **double counting** (instead of working out $c$, we work out $2c$ then divide the answer by 2).  

Consider taking one point from $n_{ij}$, i.e. in $X_i \cap Y_j$. Where can the other point be so that it counts towards $c$? It needs to be in the **same** cluster as in the ground truth, so it needs to be in $X_i$ but in a **different**  cluster to the prediction, so it can't be in $Y_j$. Graphically, the other point needs to be from the **same row** but **different column** in the contingency table. 

<center>
<img src="https://i.postimg.cc/k5v5Wb3m/Contingency-table.jpg" width="400">
</center>

For example, if the **first** point is taken from $n_{11}$, then the **second** point must be taken from any of $n_{12}, n_{13}, \ldots, n_{1s}$ so that it counts towards $c$. To work out how many **ways** there are to pick a pair of points according to this, we **multiply** $n_{11}$ (ways of chosing first point) by $(n_{12}+n_{13}+ \ldots + n_{1s})$ (ways of chosing 2nd point), to get $n_{11} (n_{12}+n_{12}+ \ldots + n_{1s})$. Note that $n_{12}+n_{13}+ \ldots + n_{1s} = a_1 - n_{11}$ (because $a_1$ is the row sum) so this **simplifies** to $n_{11} (a_1 - n_{11})$. 

As another example, if the **first** point is taken from $n_{12}$, then the other **second** point must be taken from $n_{11}, n_{13}, \ldots, n_{1s}$. Using the same reasoning as before we get that the number of ways of doing this is given by $n_{12} (a_1 - n_{12})$. Great, so now we can just **sum** over all intersections $n_{ij}$ right? Well, almost. If we do this then what happens is that we would have **double counted** all of the pairs. This is because using this scheme each point is counted as both the **first** point and the **second** point in the relevant pairs. The solution to this is easy though, we just divide by 2. 

$$
\begin{align*}
2 c &= \sum_{ij} n_{ij} (a_i - n_{ij}) \newline
&= \sum_{ij} n_{ij} a_{i} - \sum_{ij} n_{ij}^2 \newline
&= \sum_{i} \left(\sum_{j} n_{ij} \right) a_{i} -  \sum_{ij} n_{ij}^2 \newline
&= \sum_{i} a_{i}^2 -  \sum_{ij} n_{ij}^2
\end{align*}
$$

Therefore,
$$
c = \frac12 \left(\sum_{i} a_{i}^2 -  \sum_{ij} n_{ij}^2 \right)
$$

<hr>

The argument for $d$ is almost identical because of **symmetry**. All you have to do is swap the $i$'s with the $j$'s and swap the $a_i$'s with the $b_j$'s. If you want to test your understanding, this would be a really good **exercise** to derive it for yourself.

$$
d = \frac12 \left(\sum_{j} b_{j}^2 -  \sum_{ij} n_{ij}^2 \right)
$$

<hr>

And now we just need to work out $b$. There is probably a clever combinatorial argument for this, but I'm going to **cheat** a little bit by using the fact that $a+b+c+d = {n \choose 2}$ to make things easier.

$$
\begin{align*}
b &= {n \choose 2} - a - c - d \newline
&= {n \choose 2} - \sum_{i,j} {n_{ij} \choose 2} - \frac12 \left(\sum_{i} a_{i}^2 -  \sum_{ij} n_{ij}^2 \right) - \frac12 \left(\sum_{j} b_{j}^2 -  \sum_{ij} n_{ij}^2 \right) \newline
&= \frac12 \left(n^2 - n - \sum_{i,j} n_{ij}^2 + \sum_{i,j} n_{ij} - \left(\sum_{i} a_{i}^2 + \sum_{j} b_{j}^2 \right) + 2 \sum_{i,j} n_{ij}^2 \right)
\end{align*}
$$
Using the fact that $\sum_{ij} n_{ij}=n$ (there are n data points in total) and after some simplifying we get

$$
b= \frac12 \left(n^2 + \sum_{i,j} n_{ij}^2 - \left(\sum_{i} a_{i}^2 + \sum_{j} b_{j}^2 \right) \right)
$$

<hr>

So now we have **all the pieces** to work out the Rand Index using the contingency table. You will see why this helpful when we work out its expected value. For now, let's work out the number of pairs in **agreement**, i.e. $a+b$. Unfortunately, this is a bit of **messy algebra** to get it into the form we want. I've tried to break it down as much as I can. 

$$
\begin{align*}
a+b &= \sum_{i,j} {n_{ij} \choose 2} + \frac12 \left(n^2 + \sum_{i,j} n_{ij}^2 - \left(\sum_{i} a_{i}^2 + \sum_{j} b_{j}^2 \right) \right) \newline
&= \sum_{i,j} {n_{ij} \choose 2} + {n \choose 2}  + \frac{n}{2} + \frac12 \sum_{ij} n_{ij}^2 - \frac12 \left( \sum_{i} a_i^2 + \sum_{j} b_j^2 \right) \newline
&= \sum_{i,j} {n_{ij} \choose 2} + {n \choose 2}  + \sum_{i,j} {n_{ij} \choose 2} + \sum_{i,j} n_{ij} - \frac12 \left( \sum_{i} a_i^2 + \sum_{j} b_j^2 \right) \newline
&= {n \choose 2} + 2 \sum_{i,j} {n_{ij} \choose 2} + n - \frac12 \left( \sum_{i} a_i^2 + \sum_{j} b_j^2 \right) \newline
&= {n \choose 2} + 2 \sum_{i,j} {n_{ij} \choose 2} - \left( \sum_{i} {a_i \choose 2} + \sum_{j} {b_j \choose 2} \right)
\end{align*}
$$

Therefore,

$$
RI = \frac{a+b}{{n \choose 2}} = 1 + 2 \sum_{i,j} {n_{ij} \choose 2} \Big/ {n \choose 2} - \left( \sum_{i} {a_i \choose 2} + \sum_{j} {b_j \choose 2} \right) \Big/ {n \choose 2}
$$

<hr>

This is where the **magic** starts. Assuming a generalised **hypergeometric distribution** (i.e. the **probabilistic** way to model clusters) Hubert in 1977, showed the following very useful result.

$$
\mathbb{E} \left( \sum_{ij} {n_{ij} \choose 2} \right)
 = \sum_{i} {a_i \choose 2} \sum_{j} {b_j \choose 2} \Big/ {n \choose 2}
$$
 
Unfortunately, the paper where this was proved is **not open access** so I couldn't verify the details. Regardless, it allows us to calculate the expected value of the Rand Index. Combining the above two formulas gives:

$$
\mathbb{E} (RI)  = 1 + 2 \sum_{i} {a_i \choose 2} \sum_{j} {b_j \choose 2} \Big/ {n \choose 2}^2 - \left( \sum_{i} {a_i \choose 2} + \sum_{j} {b_j \choose 2} \right) \Big/ {n \choose 2}
$$

Along with the fact that $max(RI)=1$, we have everything we need to calculate the Adjusted Rand Index.

<hr>

Recall the formula,

$$
ARI = \frac{RI - \mathbb{E}(RI)}{\max(RI)-\mathbb{E}(RI)}
$$

For the numerator, we have

$$
\begin{align*}
RI - \mathbb{E}(RI) &= 1 + 2 \sum_{i,j} {n_{ij} \choose 2} \Big/ {n \choose 2} - \left( \sum_{i} {a_i \choose 2} + \sum_{j} {b_j \choose 2} \right) \Big/ {n \choose 2} \newline
&- 1 - 2 \sum_{i} {a_i \choose 2} \sum_{j} {b_j \choose 2} \Big/ {n \choose 2}^2 + \left( \sum_{i} {a_i \choose 2} + \sum_{j} {b_j \choose 2} \right) \Big/ {n \choose 2} \newline
&= 2 \sum_{i,j} {n_{ij} \choose 2} \Big/ {n \choose 2} - 2 \sum_{i} {a_i \choose 2} \sum_{j} {b_j \choose 2} \Big/ {n \choose 2}^2
\end{align*}
$$

For the denominator, we have

$$
\begin{align*}
\max(RI)-\mathbb{E}(RI) &= 1 - 1 - 2 \sum_{i} {a_i \choose 2} \sum_{j} {b_j \choose 2} \Big/ {n \choose 2}^2 + \left( \sum_{i} {a_i \choose 2} + \sum_{j} {b_j \choose 2} \right) \Big/ {n \choose 2} \newline
&= \left( \sum_{i} {a_i \choose 2} + \sum_{j} {b_j \choose 2} \right) \Big/ {n \choose 2} - 2 \sum_{i} {a_i \choose 2} \sum_{j} {b_j \choose 2} \Big/ {n \choose 2}^2
\end{align*}
$$

Finally, putting the numerator on top and denominator on the bottom and dividing both by the common factor of $2 / {n \choose 2}$, we get

$$
\large ARI = \frac{ \left. \sum_{ij} \binom{n_{ij}}{2} - \left[\sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2}\right] \right/ \binom{n}{2} }{ \left. \frac{1}{2} \left[\sum_i \binom{a_i}{2} + \sum_j \binom{b_j}{2}\right] - \left[\sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2}\right] \right/ \binom{n}{2} }
$$

as required.